# Análisis revisado — Recomendación de medicamentos (respuesta a revisores)

Notebook único y reproducible que sustituye a los notebooks 01–04. Implementa:

1. **Flujo de datos** con recuentos en cada etapa (`flujo_datos.csv`).
2. **Unidad de análisis**: cada fila es un par *episodio–medicamento*. Como el archivo integrado no conserva el `ID` original, se reconstruye un **episodio aproximado** = filas consecutivas con el mismo perfil clínico (EDAD, SEXO, TIPOURGENCIA, AFECPRIN, AFEC).
3. **Partición interna 80/20 agrupada por episodio** (`StratifiedGroupKFold`, semilla 42): ningún episodio queda repartido entre Entrenamiento y Prueba. `SPLIT='fila'` reproduce el análisis principal del artículo; `SPLIT='grupo'` el análisis de sensibilidad.
4. **Líneas base**: popularidad global, popularidad condicionada por diagnóstico principal y por diagnóstico+comorbilidad (con retroceso si hay < 5 casos).
5. **Evaluación a nivel fila** (etiqueta única) **y a nivel episodio** (acierto si cualquier medicamento del episodio aparece en el Top-K; Recall@5).
6. Detalles de K-means (k, inicialización, métricas internas, varianza SVD real) y de Apriori (búsqueda, desempates, longitud de listas, rendimiento en casos cubiertos).
7. **Comparaciones pareadas auditables**: tablas 2×2, McNemar exacto, diferencias con IC95 % bootstrap (por conglomerado de episodio) y corrección de Holm.

Todos los resultados se guardan en la carpeta `salida_<SPLIT>/` y se descargan como ZIP al final.

In [ ]:
!pip -q install mlxtend statsmodels scikit-learn pandas numpy matplotlib

In [ ]:
import io
from google.colab import files
SPLIT = 'fila'   # 'fila' = análisis principal del artículo | 'grupo' = sensibilidad agrupada por episodio
print("Sube Tacotalpa2024.csv")
up = files.upload(); CSV = next(iter(up))
OUT = f"salida_{SPLIT}"
import sys, os, json, random, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, RandomizedSearchCV, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
RS = 42
random.seed(RS); np.random.seed(RS)
os.makedirs(OUT, exist_ok=True)

In [ ]:
FEATURES = ['EDAD', 'SEXO', 'TIPOURGENCIA', 'AFECPRIN', 'AFEC']
TARGET = 'MEDICAMENTOS'
KS = (1, 3, 5)
LOG = {}

## 1. Flujo de datos

In [ ]:
# ---------------------------------------------------------------- 1. Flujo de datos
df = pd.read_csv(CSV, encoding="latin1")
flow = [("Registros en el archivo integrado (3 hospitales, 2020-2022)", len(df))]
for c in ['AFECPRIN', 'AFEC', 'MEDICAMENTOS', 'DESCRIPCION', 'SEXO', 'TIPOURGENCIA']:
    df[c] = df[c].astype("string").str.strip()
df['EDAD'] = pd.to_numeric(df['EDAD'], errors='coerce')
n0 = len(df)
df = df.dropna(subset=FEATURES + [TARGET]).copy()
flow.append(("Excluidos por valores faltantes en variables esenciales", -(n0 - len(df))))
# Episodio aproximado: filas consecutivas con la misma EDAD, SEXO, TIPOURGENCIA y AFECPRIN (el archivo
# integrado conserva el orden de la unión URGENCIAS-AFECCIONES-MEDICAMENTOS, pero no el ID original).
# AFEC no forma parte de la clave porque la unión genera el producto AFEC x MEDICAMENTO dentro de un episodio.
EP_KEY = ['EDAD', 'SEXO', 'TIPOURGENCIA', 'AFECPRIN']
key = df[EP_KEY].astype(str).agg('|'.join, axis=1)
df['EPISODIO'] = (key != key.shift()).cumsum().astype(int)
ndup = int(df.duplicated(subset=FEATURES + [TARGET, 'DESCRIPCION']).sum())
df = df.drop_duplicates(subset=FEATURES + [TARGET, 'DESCRIPCION']).reset_index(drop=True)
flow.append(("Excluidos por duplicados exactos", -ndup))
# Un medicamento repetido dentro de un mismo episodio (distinta presentación) se cuenta una vez
# para el nivel episodio; a nivel fila se conserva como en el protocolo original.
flow.append(("Conjunto analítico (filas episodio-medicamento)", len(df)))
LOG['n_analitico'] = len(df)
LOG['n_episodios'] = int(df.EPISODIO.nunique())
LOG['n_perfiles_unicos'] = int(df[FEATURES].astype(str).agg('|'.join, axis=1).nunique())
ep_size = df.groupby('EPISODIO')[TARGET].nunique()
LOG['episodios_con_1_med'] = int((ep_size == 1).sum())
LOG['episodios_con_2plus_med'] = int((ep_size >= 2).sum())
LOG['filas_en_episodios_multi'] = int(df.EPISODIO.map(df.groupby('EPISODIO').size()).gt(1).sum())
LOG['episodios_multi_afec'] = int((df.groupby('EPISODIO').AFEC.nunique() > 1).sum())
LOG['max_meds_episodio'] = int(ep_size.max())

## 2. Descripción de clases

In [ ]:
# ---------------------------------------------------------------- 2. Descripción de clases
vc = df[TARGET].value_counts()
LOG['n_clases'] = int(len(vc))
LOG['clases_1_obs'] = int((vc == 1).sum())
LOG['clases_lt5'] = int((vc < 5).sum())
LOG['clases_lt10'] = int((vc < 10).sum())
for k in (1, 3, 5, 10, 20):
    LOG[f'prop_top{k}'] = float(vc.head(k).sum() / len(df))
p = vc / vc.sum()
LOG['entropia_bits'] = float(-(p * np.log2(p)).sum())
LOG['perplejidad'] = float(2 ** LOG['entropia_bits'])
vc.rename_axis('Medicamento').reset_index(name='Frecuencia').assign(
    Porcentaje=lambda d: 100 * d.Frecuencia / len(df),
    Acumulado=lambda d: d.Porcentaje.cumsum()).to_csv(f"{OUT}/distribucion_medicamentos.csv", index=False)

## 3. Partición

In [ ]:
# ---------------------------------------------------------------- 3. Partición
singleton = vc[vc < 2].index
ep_single = set(df.loc[df[TARGET].isin(singleton), 'EPISODIO'])
if SPLIT == 'grupo':
    pool = df[~df.EPISODIO.isin(ep_single)]
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RS)
    tr_idx, te_idx = next(sgkf.split(pool, pool[TARGET], groups=pool.EPISODIO))
    test_df = pool.iloc[te_idx].copy()
    train_df = pd.concat([pool.iloc[tr_idx], df[df.EPISODIO.isin(ep_single)]]).copy()
else:  # protocolo original: estratificado por fila, sin agrupar
    df_single = df[df[TARGET].isin(singleton)]
    df_split = df[~df[TARGET].isin(singleton)]
    tr, test_df = train_test_split(df_split, test_size=.2, random_state=RS, stratify=df_split[TARGET])
    train_df = pd.concat([tr, df_single]).copy()
train_df = train_df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)
LOG['split'] = SPLIT
LOG['n_train'] = len(train_df); LOG['n_test'] = len(test_df)
LOG['ep_train'] = int(train_df.EPISODIO.nunique()); LOG['ep_test'] = int(test_df.EPISODIO.nunique())
LOG['ep_compartidos'] = len(set(train_df.EPISODIO) & set(test_df.EPISODIO))
LOG['filas_singleton_train'] = int(df[TARGET].isin(singleton).sum())
LOG['filas_en_ep_singleton_train'] = int(df.EPISODIO.isin(ep_single).sum())
LOG['clases_test_no_vistas'] = int((~test_df[TARGET].isin(set(train_df[TARGET]))).sum())
flow.append(("Entrenamiento (filas)", len(train_df)))
flow.append(("Prueba interna reservada (filas)", len(test_df)))
pd.DataFrame(flow, columns=['Etapa', 'Registros']).to_csv(f"{OUT}/flujo_datos.csv", index=False)

# Representación a nivel episodio para la prueba: un perfil y el conjunto de medicamentos.
test_ep = test_df.groupby('EPISODIO').agg(**{c: (c, 'first') for c in FEATURES},
                                           MEDS=(TARGET, lambda s: set(s))).reset_index()
LOG['test_episodios'] = len(test_ep)

## utilidades de métricas

In [ ]:
# ---------------------------------------------------------------- utilidades de métricas
def row_hits(y, recs):
    """Matriz por caso: rango (1..5) del medicamento verdadero o 0 si no aparece."""
    return np.array([(r.index(t) + 1) if t in r else 0 for t, r in zip(y, recs)])


def set_ranks(sets, recs):
    """Nivel episodio: rango del primer medicamento relevante del episodio."""
    out = []
    for s, r in zip(sets, recs):
        pos = [i + 1 for i, m in enumerate(r) if m in s]
        out.append(pos[0] if pos else 0)
    return np.array(out)


def summarize(ranks, recs, sets=None):
    d = {f'Hit@{k}': float(np.mean((ranks > 0) & (ranks <= k))) for k in KS}
    d['MRR'] = float(np.mean(np.where(ranks > 0, 1 / np.maximum(ranks, 1), 0)))
    d['Cobertura'] = float(np.mean([len(r) > 0 for r in recs]))
    if sets is not None:  # recall@5 a nivel episodio
        d['Recall@5'] = float(np.mean([len(set(r[:5]) & s) / len(s) for s, r in zip(sets, recs)]))
    return d

## 4. Líneas base

In [ ]:
# ---------------------------------------------------------------- 4. Líneas base
glob = train_df[TARGET].value_counts().index.tolist()


def cond_pop(cols, min_n=1):
    tab = train_df.groupby(cols + [TARGET]).size().rename('n').reset_index()
    tab = tab.sort_values(cols + ['n', TARGET], ascending=[True] * len(cols) + [False, True])
    return {k if isinstance(k, tuple) else (k,): (g[TARGET].tolist(), int(g.n.sum())) for k, g in tab.groupby(cols)}


pop_dx = cond_pop(['AFECPRIN'])
pop_dx2 = cond_pop(['AFECPRIN', 'AFEC'])


MIN_N = 5  # mínimo de casos de entrenamiento para usar un perfil condicionado; si no, se retrocede


def rec_backoff(row, tables, k=5):
    out = []
    for cols, tab in tables:
        lst, n = tab.get(tuple(row[c] for c in cols), ([], 0))
        if n < MIN_N:
            continue
        for m in lst:
            if m not in out:
                out.append(m)
            if len(out) == k:
                return out
    for m in glob:
        if m not in out:
            out.append(m)
        if len(out) == k:
            return out
    return out


RECS = {}
def predict_all(fn):
    rows = [fn(r) for _, r in test_df.iterrows()]
    eps = [fn(r) for _, r in test_ep.iterrows()]
    return rows, eps

RECS['Popularidad global'] = predict_all(lambda r: glob[:5])
RECS['Popularidad por diagnóstico'] = predict_all(lambda r: rec_backoff(r, [(['AFECPRIN'], pop_dx)]))
RECS['Popularidad por diagnóstico+comorbilidad'] = predict_all(
    lambda r: rec_backoff(r, [(['AFECPRIN', 'AFEC'], pop_dx2), (['AFECPRIN'], pop_dx)]))
LOG['test_dx_no_visto'] = int((~test_df.AFECPRIN.isin(set(train_df.AFECPRIN))).sum())

## 5. Árbol de decisión

In [ ]:
# ---------------------------------------------------------------- 5. Árbol de decisión
cat_cols = ['SEXO', 'TIPOURGENCIA', 'AFECPRIN', 'AFEC']
pre_tree = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
                              ('num', 'passthrough', ['EDAD'])])
pipe = Pipeline([('preprocess', pre_tree), ('classifier', DecisionTreeClassifier(random_state=RS))])
param_dist = {'classifier__criterion': ['gini', 'entropy', 'log_loss'],
              'classifier__max_depth': [5, 10, 15, 20, 30, None],
              'classifier__min_samples_split': [2, 5, 10, 20],
              'classifier__min_samples_leaf': [1, 2, 5, 10],
              'classifier__ccp_alpha': [0.0, 0.0001, 0.001, 0.005, 0.01],
              'classifier__class_weight': [None, 'balanced']}
cv = GroupKFold(n_splits=3) if SPLIT == 'grupo' else __import__('sklearn.model_selection', fromlist=['KFold']).KFold(3, shuffle=True, random_state=RS)
search = RandomizedSearchCV(pipe, param_dist, n_iter=24, scoring='f1_weighted', cv=cv, n_jobs=-1,
                            random_state=RS, error_score=0.0)
if SPLIT == 'grupo':
    search.fit(train_df[FEATURES], train_df[TARGET], groups=train_df.EPISODIO)
else:
    search.fit(train_df[FEATURES], train_df[TARGET])
tree = search.best_estimator_
LOG['tree_best_params'] = {k.replace('classifier__', ''): (v if v is not None else 'None') for k, v in search.best_params_.items()}
LOG['tree_cv_f1w'] = float(search.best_score_)
LOG['tree_depth'] = int(tree.named_steps['classifier'].get_depth())
LOG['tree_leaves'] = int(tree.named_steps['classifier'].get_n_leaves())
classes = tree.named_steps['classifier'].classes_


def tree_top5(X):
    pr = tree.predict_proba(X[FEATURES])
    # ranking por probabilidad descendente (mismo procedimiento que el notebook original)
    top = np.argsort(pr, axis=1)[:, ::-1][:, :5]
    return [[classes[j] for j in row] for row in top]


RECS['Árbol de decisión'] = (tree_top5(test_df), tree_top5(test_ep))


def hit3_scorer(est, X, y_true):
    pr = est.predict_proba(X); cl = est.classes_
    top = np.argsort(-pr, axis=1)[:, :3]
    return float(np.mean([t in set(cl[j] for j in row) for t, row in zip(np.asarray(y_true), top)]))


search3 = RandomizedSearchCV(pipe, param_dist, n_iter=24, scoring=hit3_scorer, cv=cv, n_jobs=-1,
                             random_state=RS, error_score=0.0)
if SPLIT == 'grupo':
    search3.fit(train_df[FEATURES], train_df[TARGET], groups=train_df.EPISODIO)
else:
    search3.fit(train_df[FEATURES], train_df[TARGET])
LOG['tree_hit3_best_params'] = {k.replace('classifier__', ''): (v if v is not None else 'None') for k, v in search3.best_params_.items()}
LOG['tree_hit3_cv'] = float(search3.best_score_)
_tree_main = tree
tree = search3.best_estimator_
RECS['Árbol de decisión (ajuste por Hit@3)'] = (tree_top5(test_df), tree_top5(test_ep))
tree = _tree_main

## 6. K-means

In [ ]:
# ---------------------------------------------------------------- 6. K-means
pre_km = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
                            ('age', StandardScaler(with_mean=False), ['EDAD'])])
Xtr_s = pre_km.fit_transform(train_df[FEATURES])
LOG['km_dim_onehot'] = int(Xtr_s.shape[1])
svd = TruncatedSVD(n_components=min(50, Xtr_s.shape[1] - 1), random_state=RS)
Xtr = svd.fit_transform(Xtr_s)
LOG['svd_var_50'] = float(svd.explained_variance_ratio_.sum())
# componentes necesarios para 80/90/95 % (SVD amplio sólo para el reporte)
svd_big = TruncatedSVD(n_components=min(600, Xtr_s.shape[1] - 1), random_state=RS).fit(Xtr_s)
cum = np.cumsum(svd_big.explained_variance_ratio_)
for th in (0.8, 0.9, 0.95):
    LOG[f'svd_comp_{int(th*100)}'] = int(np.searchsorted(cum, th) + 1) if cum[-1] >= th else None
pd.DataFrame({'componentes': np.arange(1, len(cum) + 1), 'varianza_acumulada': cum}).to_csv(
    f"{OUT}/svd_varianza_acumulada.csv", index=False)
rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=RS)
    lab = km.fit_predict(Xtr)
    rows.append({'k': k, 'Silhouette': silhouette_score(Xtr, lab, sample_size=min(3000, len(Xtr)), random_state=RS),
                 'DaviesBouldin': davies_bouldin_score(Xtr, lab), 'CalinskiHarabasz': calinski_harabasz_score(Xtr, lab),
                 'Inercia': km.inertia_})
kt = pd.DataFrame(rows)
kt['rank_promedio'] = (kt.Silhouette.rank(ascending=False) + kt.DaviesBouldin.rank() + kt.CalinskiHarabasz.rank(ascending=False)) / 3
kt.to_csv(f"{OUT}/kmeans_seleccion_k.csv", index=False)
BEST_K = int(kt.sort_values(['rank_promedio', 'k']).iloc[0].k)
LOG['km_k'] = BEST_K
LOG['km_scores'] = kt[kt.k == BEST_K].iloc[0].drop('k').to_dict()
kmeans = KMeans(n_clusters=BEST_K, init='k-means++', n_init=20, random_state=RS).fit(Xtr)
tr_cl = kmeans.labels_
LOG['km_cluster_sizes'] = pd.Series(tr_cl).value_counts().sort_index().tolist()
cl_rank = {c: train_df.loc[tr_cl == c, TARGET].value_counts().index.tolist() for c in range(BEST_K)}


def km_top5(X):
    cl = kmeans.predict(svd.transform(pre_km.transform(X[FEATURES])))
    return [cl_rank[int(c)][:5] for c in cl]


RECS['K-means'] = (km_top5(test_df), km_top5(test_ep))

## 7. Apriori

In [ ]:
# ---------------------------------------------------------------- 7. Apriori
def age_band(a):
    for lo, hi, lab in [(0, 11, '00_11'), (12, 17, '12_17'), (18, 29, '18_29'), (30, 44, '30_44'),
                        (45, 59, '45_59'), (60, 74, '60_74'), (75, 200, '75_MAS')]:
        if lo <= a <= hi:
            return lab
    return 'DESCONOCIDA'


def tok(p, v):
    return f"{p}={str(v).strip().replace('|', '/')}"


def transaction(r, med=True):
    x = [tok('EDADGRUPO', age_band(float(r['EDAD']))), tok('SEXO', r['SEXO']), tok('URGENCIA', r['TIPOURGENCIA']),
         tok('AFECPRIN', r['AFECPRIN']), tok('AFEC', r['AFEC'])]
    if med:
        x.append(tok('MED', r[TARGET]))
    return x


def make_rules(data, s, c, max_len=4):
    tx = [transaction(r) for _, r in data.iterrows()]
    te = TransactionEncoder(); basket = pd.DataFrame(te.fit(tx).transform(tx), columns=te.columns_)
    freq = apriori(basket, min_support=s, use_colnames=True, max_len=max_len, low_memory=True)
    if freq.empty:
        return pd.DataFrame(), 0
    try:
        r = association_rules(freq, metric='confidence', min_threshold=c)
    except Exception:
        return pd.DataFrame(), len(freq)
    if r.empty:
        return r, len(freq)
    ok = r.apply(lambda z: len(z.consequents) == 1 and next(iter(z.consequents)).startswith('MED=')
                 and all(not q.startswith('MED=') for q in z.antecedents) and z.lift > 1, axis=1)
    r = r[ok].sort_values(['confidence', 'lift', 'support'], ascending=False).reset_index(drop=True)
    return r, len(freq)


def ap_recommend(row, rules, k=5):
    if rules is None or rules.empty:
        return []
    patient = set(transaction(row, False)); best = {}
    for z in rules.itertuples():
        if set(z.antecedents) <= patient:
            med = next(iter(z.consequents)).replace('MED=', '', 1)
            score = (z.confidence, z.lift, z.support, len(z.antecedents))
            if med not in best or score > best[med]:
                best[med] = score
    # orden: confianza, lift, soporte, longitud del antecedente; desempate final alfabético
    return [m for m, _ in sorted(best.items(), key=lambda x: (tuple(-v for v in x[1]), x[0]))[:k]]


# Validación interna dentro de Train (agrupada por episodio)
if SPLIT == 'grupo':
    vc_tr = train_df[TARGET].value_counts()
    ep_s = set(train_df.loc[train_df[TARGET].isin(vc_tr[vc_tr < 2].index), 'EPISODIO'])
    pool = train_df[~train_df.EPISODIO.isin(ep_s)]
    a, b = next(StratifiedGroupKFold(5, shuffle=True, random_state=RS).split(pool, pool[TARGET], pool.EPISODIO))
    subtrain = pd.concat([pool.iloc[a], train_df[train_df.EPISODIO.isin(ep_s)]]); val = pool.iloc[b]
else:
    vc_tr = train_df[TARGET].value_counts(); spl = vc_tr[vc_tr >= 2].index
    tsp = train_df[train_df[TARGET].isin(spl)]
    sm, val = train_test_split(tsp, test_size=.2, random_state=RS, stratify=tsp[TARGET])
    subtrain = pd.concat([sm, train_df[~train_df[TARGET].isin(spl)]])
grid = []
for s in [.001, .0025, .005, .01]:
    for c in [.30, .40, .50, .60, .70]:
        rl, ni = make_rules(subtrain, s, c)
        rc = [ap_recommend(r, rl) for _, r in val.iterrows()]
        grid.append({'min_support': s, 'min_confidence': c, 'itemsets': ni, 'reglas': len(rl),
                     **summarize(row_hits(val[TARGET].tolist(), rc), rc)})
grid = pd.DataFrame(grid).sort_values(['Hit@3', 'Cobertura', 'MRR', 'Hit@1', 'min_confidence'],
                                      ascending=False).reset_index(drop=True)
grid.to_csv(f"{OUT}/apriori_busqueda.csv", index=False)
S, C = float(grid.iloc[0].min_support), float(grid.iloc[0].min_confidence)
rules, nit = make_rules(train_df, S, C)
LOG['ap_support'] = S; LOG['ap_conf'] = C; LOG['ap_itemsets'] = nit; LOG['ap_rules'] = len(rules)
rx = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
rx['antecedents'] = rx.antecedents.apply(lambda x: ' & '.join(sorted(x)))
rx['consequents'] = rx.consequents.apply(lambda x: ' & '.join(sorted(x)))
rx.to_csv(f"{OUT}/apriori_reglas.csv", index=False)
RECS['Apriori'] = ([ap_recommend(r, rules) for _, r in test_df.iterrows()],
                   [ap_recommend(r, rules) for _, r in test_ep.iterrows()])

## 8. Resultados

In [ ]:
# ---------------------------------------------------------------- 8. Resultados
y = test_df[TARGET].tolist(); sets = test_ep.MEDS.tolist()
RANK_ROW = {m: row_hits(y, RECS[m][0]) for m in RECS}
RANK_EP = {m: set_ranks(sets, RECS[m][1]) for m in RECS}
groups_row = test_df.EPISODIO.values
B = 2000
rng = np.random.default_rng(RS)
ug = np.unique(groups_row); gi = {g: np.where(groups_row == g)[0] for g in ug}
BOOT_ROW = [np.concatenate([gi[g] for g in rng.choice(ug, len(ug))]) for _ in range(B)]  # bootstrap por conglomerado
n_ep = len(test_ep)
BOOT_EP = [rng.integers(0, n_ep, n_ep) for _ in range(B)]


def metric_vec(ranks):
    return {**{f'Hit@{k}': ((ranks > 0) & (ranks <= k)).astype(float) for k in KS},
            'MRR': np.where(ranks > 0, 1 / np.maximum(ranks, 1), 0.0)}


def table(level):
    RK, RC, BO = (RANK_ROW, 0, BOOT_ROW) if level == 'fila' else (RANK_EP, 1, BOOT_EP)
    rows = []
    for m in RECS:
        base = summarize(RK[m], RECS[m][RC], sets if level == 'episodio' else None)
        mv = metric_vec(RK[m])
        for met, v in mv.items():
            bs = [v[ix].mean() for ix in BO]
            base[met + '_IC95'] = f"[{100*np.quantile(bs,.025):.2f}, {100*np.quantile(bs,.975):.2f}]" if met != 'MRR' \
                else f"[{np.quantile(bs,.025):.4f}, {np.quantile(bs,.975):.4f}]"
        cov = np.array([len(r) > 0 for r in RECS[m][RC]])
        if cov.mean() < 1:
            for k in KS:
                base[f'Hit@{k}_cubiertos'] = float(((RK[m] > 0) & (RK[m] <= k))[cov].mean())
            base['MRR_cubiertos'] = float(mv['MRR'][cov].mean())
        rows.append({'Modelo': m, 'n': len(RK[m]), **base})
    return pd.DataFrame(rows)


res_row = table('fila'); res_ep = table('episodio')
res_row.to_csv(f"{OUT}/resultados_nivel_fila.csv", index=False)
res_ep.to_csv(f"{OUT}/resultados_nivel_episodio.csv", index=False)

# Distribución de longitud de lista Apriori
for lvl, RC in (('fila', 0), ('episodio', 1)):
    ln = pd.Series([len(r) for r in RECS['Apriori'][RC]]).value_counts().sort_index()
    pd.DataFrame({'longitud_lista': ln.index, 'casos': ln.values, 'porcentaje': 100 * ln.values / ln.sum()}).to_csv(
        f"{OUT}/apriori_longitud_lista_{lvl}.csv", index=False)

# Comparaciones pareadas: McNemar exacto (Hit@1) + diferencias con IC bootstrap para todas las métricas
pairs = [('Árbol de decisión', x) for x in ['K-means', 'Apriori', 'Popularidad global', 'Popularidad por diagnóstico',
                                            'Popularidad por diagnóstico+comorbilidad']] + [('K-means', 'Apriori'), ('K-means', 'Popularidad global')]
# Familia de Holm: 7 pares x 3 métricas Hit@K = 21 pruebas McNemar


def paired(level):
    RK, BO = (RANK_ROW, BOOT_ROW) if level == 'fila' else (RANK_EP, BOOT_EP)
    out = []
    for a, b in pairs:
        va, vb = metric_vec(RK[a]), metric_vec(RK[b])
        for k in KS:
            ha, hb = va[f'Hit@{k}'].astype(int), vb[f'Hit@{k}'].astype(int)
            t = [[int(((ha == 1) & (hb == 1)).sum()), int(((ha == 1) & (hb == 0)).sum())],
                 [int(((ha == 0) & (hb == 1)).sum()), int(((ha == 0) & (hb == 0)).sum())]]
            p = mcnemar(t, exact=True).pvalue
            d = va[f'Hit@{k}'] - vb[f'Hit@{k}']
            bs = [d[ix].mean() for ix in BO]
            out.append({'A': a, 'B': b, 'Métrica': f'Hit@{k}', 'Ambos aciertan': t[0][0], 'Sólo A': t[0][1],
                        'Sólo B': t[1][0], 'Ninguno': t[1][1], 'Diferencia (pp)': 100 * d.mean(),
                        'IC95 inf (pp)': 100 * np.quantile(bs, .025), 'IC95 sup (pp)': 100 * np.quantile(bs, .975),
                        'OR discordante (b/c)': (t[0][1] / t[1][0]) if t[1][0] else np.inf, 'p_McNemar': p})
        d = va['MRR'] - vb['MRR']; bs = [d[ix].mean() for ix in BO]
        out.append({'A': a, 'B': b, 'Métrica': 'MRR', 'Diferencia (pp)': d.mean(), 'IC95 inf (pp)': np.quantile(bs, .025),
                    'IC95 sup (pp)': np.quantile(bs, .975),
                    'p_McNemar': np.nan})
    o = pd.DataFrame(out)
    m = o.p_McNemar.notna()
    o.loc[m, 'p_Holm'] = multipletests(o.loc[m, 'p_McNemar'], method='holm')[1]
    return o


paired('fila').to_csv(f"{OUT}/comparaciones_pareadas_fila.csv", index=False)
paired('episodio').to_csv(f"{OUT}/comparaciones_pareadas_episodio.csv", index=False)

# Predicciones
pd.DataFrame({'EPISODIO': test_df.EPISODIO, 'Real': y,
              **{f'Top5_{m}': [' | '.join(r) for r in RECS[m][0]] for m in RECS}}).to_csv(f"{OUT}/predicciones_test_fila.csv", index=False)
with open(f"{OUT}/resumen.json", 'w', encoding='utf-8') as f:
    json.dump(LOG, f, ensure_ascii=False, indent=2, default=float)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print(json.dumps(LOG, ensure_ascii=False, indent=1, default=float))
print(pd.DataFrame(flow))
print(res_row.round(4).to_string()); print(res_ep.round(4).to_string())

## Tablas principales

In [ ]:
display(pd.read_csv(f"{OUT}/flujo_datos.csv"))
display(res_row.round(4)); display(res_ep.round(4))
display(pd.read_csv(f"{OUT}/comparaciones_pareadas_fila.csv").round(4))
display(kt.round(4)); display(grid.head(10))

## Figura de resultados y descarga

In [ ]:
import matplotlib.pyplot as plt
r = res_row[res_row.Modelo != 'Árbol de decisión (ajuste por Hit@3)']
order = ['Popularidad global','Popularidad por diagnóstico','Popularidad por diagnóstico+comorbilidad','Apriori','K-means','Árbol de decisión']
col = ['#d0cfc8','#8f8e86','#5c5b55','#1baf7a','#eb6834','#2a78d6']
fig, ax = plt.subplots(figsize=(8,4.5)); w=.13; x=np.arange(3); mets=['Hit@1','Hit@3','Hit@5']
for i,(m,c) in enumerate(zip(order,col)):
    row=r[r.Modelo==m].iloc[0]; v=[100*row[k] for k in mets]
    ax.bar(x+(i-2.5)*(w+.012), v, w, color=c, hatch='//' if i<3 else '', edgecolor='white', label=m)
ax.set_xticks(x); ax.set_xticklabels(mets); ax.set_ylabel('Aciertos en prueba (%)'); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.savefig(f"{OUT}/figura_resultados.png", dpi=300); plt.show()
import shutil; shutil.make_archive(OUT, 'zip', OUT); files.download(OUT + '.zip')